# QELM on the measurement record M, sweep over p

Experiment `experiments/QELM_M/exp_QELM_M.py` (issue #94). For each sample, the reservoir (disordered TFIM, with an ancilla entangled with the charge of the system) evolves for `evolution.t`, split into `Ns` steps. After each step, each system qubit is measured in Z with probability `p`. The features are the measurement record `M` (0 for an unmeasured qubit), and the target is `<Z_0>` of the ancilla, never measured, at the final time.

This notebook launches the sweep with `experiments/QELM_M/exp_QELM_M.sh`, reads back the folders `outputs/QELM_M/<date_time>/N<N>_seed<seed>_p<p>/`, then plots against `p` the R2, the NMSE, the accuracy of the sign of `<Z_0>` and the entropies (in bits): that of the ancilla with the system (von Neumann `S_vN` and Rényi of order 1 `S_renyi_1`, issue #119), that of the system sector without the ancilla (von Neumann `S_vN_sys` and Rényi of order 2 `S_renyi_2_sys`) and the mean over the system qubits of the entropy of each single qubit (`S_vN_1q`, `S_renyi_2_1q`), issue #124. Each metric is averaged over the seeds: on the left the mean with the standard error as error bars, on the right the standard error alone.

At p = 1, measuring the whole system fixes the charge, so `<Z_0>` is ±1 and can be read off the last row of M: R2 ≈ 1 is expected there whatever the dynamics (issue #104).

The ancilla entropy is the purification order parameter of the measurement-induced transition: `S = 1` bit as long as the ancilla stays entangled with the system, `S = 0` once the measurements have purified it. Rényi of order 1 is the `n -> 1` limit of the Rényi family, that is the von Neumann entropy: the two curves coincide.

The global state stays pure (unitary evolution and projective measurements), so the entropy of the system sector equals that of the ancilla, at every Rényi order: the `S_vN_sys` curve reproduces `S_vN`, and `S_renyi_2_sys` also gives the Rényi-2 entropy of the ancilla (`S_2 <= S_vN`). The single-qubit averages `S_vN_1q` and `S_renyi_2_1q` are new information: the entanglement of each system qubit with all the rest.

The accuracy classifies each test sample by the sign of `<Z_0>` (ancilla closer to `|0>` or to `|1>`) and counts the fraction of signs that the prediction gets right.

The two signs are not balanced (issue #130). At `h = 10 J`, the TFIM almost conserves the charge: the ancilla shares `Ns // 2` excitations with the `N - 1` other qubits, so it relaxes closer to `|0>` and `<Z_0> > 0` in most samples (about 85 % at N = 4, mean `<Z_0>` close to `1 - 2 (Ns // 2) / N`). A readout that learns nothing, as at p = 0 where every feature is 0, always predicts that majority sign and scores its share, not 0.5. The chance level of the accuracy is therefore `accuracy_baseline`, the accuracy of a classifier that always predicts the majority sign of the training set. `balanced_accuracy` averages, over the two signs, the fraction recovered: it is 0.5 for a readout that ignores the features, whatever the imbalance.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_m_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the total number of qubits, ancilla included, and `L` the number of samples per run (`samples.L` in the config). Each sweep goes to this notebook's own folder, `outputs/QELM_M/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

In [ ]:
L = os.environ.get("L", "500")  # samples per run (samples.L)
N_VALUES = os.environ.get("N_VALUES", "4")  # total qubits N, ancilla included
SEEDS = os.environ.get("SEEDS", "42 43 44")
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.5 0.8 1.0")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151): no other notebook writes
# there, so the cells below never read another notebook's sweep.
EXPERIMENT = "QELM_M"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(P_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label, log scale). A metric missing from an older
# sweep is left out: the ancilla entropies came with issue #119, the
# accuracy and the system entropies with issue #124, the accuracy baseline
# and the balanced accuracy with issue #130.
METRICS = [
    ("r2", "R2 (test)", False),
    ("nmse", "NMSE (test)", True),
    ("accuracy", "accuracy of sign <Z_0> (test)", False),
    ("accuracy_baseline", "chance level of accuracy (majority sign)", False),
    ("balanced_accuracy", "balanced accuracy of sign <Z_0> (test)", False),
    ("S_vN_mean", "S_vN ancilla (bits)", False),
    ("S_renyi_1_mean", "S_Rényi-1 ancilla (bits)", False),
    ("S_vN_sys_mean", "S_vN system (bits)", False),
    ("S_renyi_2_sys_mean", "S_Rényi-2 system (bits)", False),
    ("S_vN_1q_mean", "mean single-qubit S_vN (bits)", False),
    ("S_renyi_2_1q_mean", "mean single-qubit S_Rényi-2 (bits)", False),
]
rows = load_sweep(sweep_dir)
METRICS = [m for m in METRICS if all(m[0] in r for r in rows)]
table = summarize(rows, by=("N", "p"), metrics=[key for key, _, _ in METRICS])
print(f"{len(rows)} runs")
print(
    f"{'N':>3} {'p':>5} {'runs':>4} " + " ".join(f"{key:>20}" for key, _, _ in METRICS)
)
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.4f} ± {r[f'{key}_sem']:<8.4f}" for key, _, _ in METRICS
    )
    print(f"{r['N']:>3} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

## Metrics against p

One row per metric: on the left the mean over the seeds ± the standard error (`std / sqrt(n_runs)`), on the right the standard error alone. With a single seed the error is undefined and the right plot stays empty.

On the accuracy plot, the dashed curve of the same color is its chance level (`accuracy_baseline`, majority sign): only the gap between the two measures what the readout learns from M. On the balanced-accuracy plot, the dotted gray line marks chance, 0.5.

The figure and the data it shows are saved in the sweep folder, as `metrics_vs_p.png` and `metrics_vs_p.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

fig, axes = subplots(len(METRICS), 2, figsize=(12, 4.2 * len(METRICS)), squeeze=False)
Ns = sizes(table)
colors = color_map([f"N{N}" for N in Ns])
for N in Ns:
    color = colors[f"N{N}"]
    for (key, label, log), (ax_mean, ax_err) in zip(METRICS, axes, strict=True):
        curve = sweep_curve(table, N, "p", key)
        draw_curve(ax_mean, curve, color, label=f"N = {N}", log=log)
        sem = [r[f"{key}_sem"] for r in curve.rows]
        ax_err.plot(curve.x, sem, color=color, marker="o", label=f"N = {N}")
        # Chance level of the accuracy (issue #130): the majority-sign
        # classifier, dashed in the color of the same N.
        if key == "accuracy" and "accuracy_baseline_mean" in curve.rows[0]:
            ax_mean.plot(
                curve.x,
                [r["accuracy_baseline_mean"] for r in curve.rows],
                linestyle="--",
                color=color,
                label=f"N = {N}, chance (majority sign)",
            )
for (key, label, _), (ax_mean, ax_err) in zip(METRICS, axes, strict=True):
    # A readout that ignores the features scores 0.5 in balanced accuracy.
    if key == "balanced_accuracy":
        ax_mean.axhline(0.5, color="gray", linestyle=":", label="chance")
    ax_mean.set(xlabel="measurement probability p", ylabel=label, title=label)
    ax_err.set(
        xlabel="measurement probability p",
        ylabel=f"standard error of {key}",
        title=f"Error on {label}",
    )
    for ax in (ax_mean, ax_err):
        ax.legend()
fig.suptitle("QELM on the measurement record M")
figure_path = save_figure(fig, sweep_dir, "metrics_vs_p", data={"table": table})
print(f"figure: {figure_path}")
plt.show()